# Construction du dataset 2017

Le fichier brut du Ministère de l'Intérieur (`data_brut/presidentielle_2017_raw.xlsx`) donne
les résultats du 1er tour **par bureau de vote** (69 000 lignes), avec un format "large" :
les 11 candidats sont en colonnes répétées (N°Panneau, Sexe, Nom, Prénom, Voix... pour chacun).

Le but ici est de transformer ça en un fichier propre **par région**, au même format que 2012 et 2022.

Étapes :
1. Lire le fichier brut
2. Récupérer les voix de chaque candidat (parsing par position)
3. Rattacher chaque département à sa région
4. Agréger par région
5. Calculer les pourcentages
6. Vérifier et exporter

Note : ce fichier brut ne contient que la métropole (pas la Corse ni l'Outre-mer).
Ces deux zones ont été ajoutées séparément dans le fichier final.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('data_clean', exist_ok=True)

## Correspondance département → région

On utilise les 13 régions actuelles (réforme de 2015).

In [ ]:
dep_region = {}
def ajouter(codes, nom):
    for c in codes.split():
        dep_region[c] = nom

ajouter('01 03 07 15 26 38 42 43 63 69 73 74', 'Auvergne-Rhône-Alpes')
ajouter('21 25 39 58 70 71 89 90', 'Bourgogne-Franche-Comté')
ajouter('22 29 35 56', 'Bretagne')
ajouter('18 28 36 37 41 45', 'Centre-Val de Loire')
ajouter('08 10 51 52 54 55 57 67 68 88', 'Grand Est')
ajouter('02 59 60 62 80', 'Hauts-de-France')
ajouter('75 77 78 91 92 93 94 95', 'Île-de-France')
ajouter('14 27 50 61 76', 'Normandie')
ajouter('16 17 19 23 24 33 40 47 64 79 86 87', 'Nouvelle-Aquitaine')
ajouter('09 11 12 30 31 32 34 46 48 65 66 81 82', 'Occitanie')
ajouter('44 49 53 72 85', 'Pays de la Loire')
ajouter('04 05 06 13 83 84', "Provence-Alpes-Côte d'Azur")

# code région (pour le fichier final)
code_region = {
    'Auvergne-Rhône-Alpes': 84, 'Bourgogne-Franche-Comté': 27, 'Bretagne': 53,
    'Centre-Val de Loire': 24, 'Grand Est': 44, 'Hauts-de-France': 32,
    'Île-de-France': 11, 'Normandie': 28, 'Nouvelle-Aquitaine': 75,
    'Occitanie': 76, 'Pays de la Loire': 52, "Provence-Alpes-Côte d'Azur": 93,
}

print(len(dep_region), 'départements rattachés à', len(code_region), 'régions')

## Lecture du fichier brut

In [ ]:
# on lit sans en-tête pour traiter les colonnes par position
brut = pd.read_excel('data_brut/presidentielle_2017_raw.xlsx', header=None, skiprows=1)
print('Fichier brut :', brut.shape, '(lignes = bureaux de vote, colonnes = données)')

# rattacher la région à partir du code département (colonne 0)
def code_dep(x):
    s = str(x).strip()
    if s.endswith('.0'):
        s = s[:-2]
    return s.zfill(2)

brut['Region'] = brut[0].map(code_dep).map(dep_region)
print('Bureaux rattachés à une région :', brut['Region'].notna().sum())

## Récupération des voix par candidat

Les 8 premières colonnes sont communes (code dép, libellé, inscrits, abstentions, votants,
blancs, nuls, exprimés). Ensuite chaque candidat occupe un bloc de 7 colonnes, et le nom
est toujours à la 3e position du bloc, les voix à la 5e.

In [ ]:
# colonnes communes (par position)
communes = {'Inscrits': 2, 'Abstentions': 3, 'Votants': 4, 'Blancs': 5, 'Nuls': 6, 'Exprimes': 7}

agg = pd.DataFrame()
agg['Region'] = sorted(code_region.keys())
agg = agg.set_index('Region')
for nom, idx in communes.items():
    agg[nom] = brut.groupby('Region')[idx].sum()

# candidats : blocs de 7 colonnes à partir de la colonne 8
candidats = []
k = 0
while 8 + k * 7 + 4 < brut.shape[1] - 1:
    base = 8 + k * 7
    nom = str(brut.iloc[0, base + 2]).strip().title()
    if nom in ('Nan', '', 'None'):
        break
    brut['_voix'] = pd.to_numeric(brut.iloc[:, base + 4], errors='coerce')
    agg[nom] = brut.groupby('Region')['_voix'].sum()
    candidats.append(nom)
    k += 1

print('Candidats récupérés :', candidats)
agg.head()

## Calcul des pourcentages

In [ ]:
df = agg.reset_index()
df['Code_region'] = df['Region'].map(code_region)

# % de chaque candidat sur les exprimés
for c in candidats:
    df[f'% {c}'] = (df[c] / df['Exprimes'] * 100).round(2)

# remettre les colonnes dans un ordre propre
cols = ['Code_region', 'Region', 'Inscrits', 'Abstentions', 'Votants', 'Blancs', 'Nuls', 'Exprimes']
cols += candidats + [f'% {c}' for c in candidats]
df = df[cols].sort_values('Code_region').reset_index(drop=True)

print('Résultat :', df.shape)
df[['Region', 'Inscrits', 'Exprimes', 'Melenchon', '% Melenchon']]

## Vérification

On vérifie que les totaux nationaux (métropole) correspondent bien aux chiffres officiels 2017.

In [ ]:
total_inscrits = df['Inscrits'].sum()
total_mel = df['Melenchon'].sum()
total_exp = df['Exprimes'].sum()

print(f'Inscrits métropole   : {total_inscrits:,}')
print(f'Exprimés métropole   : {total_exp:,}')
print(f'Mélenchon métropole  : {total_mel:,}  ({total_mel/total_exp*100:.2f} %)')
print()
print('Score national officiel de Mélenchon en 2017 : 19,58 % (France entière)')
print('→ léger écart normal car ici on a la métropole sans Corse ni Outre-mer')

## Export

On ajoute la Corse et l'Outre-mer (qui ne sont pas dans le fichier brut métropole) en les
reprenant du fichier final déjà constitué, puis on exporte.

In [ ]:
sortie = 'data_clean/elections_2017_clean.xlsx'

# récupérer Corse + Outre-mer depuis le fichier existant s'il est là
try:
    ancien = pd.read_excel(sortie)
    extra = ancien[ancien['Region'].isin(['Corse', 'Outre-mer'])]
    df_final = pd.concat([df, extra[df.columns.intersection(extra.columns)]], ignore_index=True)
except FileNotFoundError:
    df_final = df
    print('(fichier existant non trouvé, on exporte la métropole seule)')

df_final.to_excel(sortie, index=False)
print('✅ Exporté :', sortie)
print('  ', len(df_final), 'régions ×', len(df_final.columns), 'colonnes')